# 02 - Data Cleaning and Standardization

## Core Question

How can the raw NFL datasets be transformed into consistent, reliable, and joinable data sources for feature engineering and modeling?

## Purpose

This notebook cleans and standardizes the raw datasets collected in `01_Data_Collection.ipynb`.

The goal is to create trustworthy intermediate datasets before any predictive features are engineered.

Key cleaning priorities include:

- Standardizing NFL team abbreviations across data sources and seasons
- Separating regular season and postseason records where appropriate
- Standardizing season and week fields
- Preserving stable player identifiers
- Identifying duplicate records
- Understanding missing data patterns
- Handling differences in historical data coverage
- Creating consistent data types
- Validating row counts and key relationships
- Saving cleaned datasets for downstream feature engineering

No predictive modeling or football ratings are created in this notebook.

# Load Raw Data

The raw datasets collected in Notebook 01 are loaded from disk without modification.

Keeping raw and cleaned data separate ensures that every cleaning step can be reproduced and audited.

In [70]:
import polars as pl
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
ROSTER_DIR = DATA_DIR / "roster"
SCHEDULE_DIR = DATA_DIR / "schedules"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

c:\Users\efriedman\Desktop\NFL-Season-Projections


In [71]:
SEASONS = list(range(2015, 2026))

schedules = pl.read_parquet(
    SCHEDULE_DIR / "nfl_schedules_2015_2025.parquet"
)

rosters = pl.read_parquet(
    ROSTER_DIR / "nfl_rosters_2015_2025.parquet"
)

player_stats = pl.read_parquet(
    RAW_DIR / "player_stats_2015_2025.parquet"
)

participation = pl.read_parquet(
    RAW_DIR / "participation_2016_2025.parquet"
)

snap_counts = pl.read_parquet(
    RAW_DIR / "snap_counts_2015_2025.parquet"
)

depth_charts = pl.read_parquet(
    RAW_DIR / "depth_charts_raw.parquet"
)

draft_picks = pl.read_parquet(
    RAW_DIR / "draft_picks_2015_2025.parquet"
)

injuries = pl.read_parquet(
    RAW_DIR / "injuries_2015_2025.parquet"
)

contracts = pl.read_parquet(
    RAW_DIR / "contracts_raw.parquet"
)

trades = pl.read_parquet(
    RAW_DIR / "trades_raw.parquet"
)

weekly_rosters = pl.read_parquet(
    RAW_DIR / "weekly_rosters_2015_2025.parquet"
)

print("All non-play-by-play raw datasets loaded successfully.")

All non-play-by-play raw datasets loaded successfully.


# Standardize Team Abbreviations

NFL datasets do not always use the same team abbreviations across sources or seasons.

A canonical abbreviation system is defined here so every downstream dataset uses one consistent team identifier.

In [72]:
TEAM_MAP = {
    # Arizona
    "ARZ": "ARI",

    # Baltimore
    "BLT": "BAL",

    # Cleveland
    "CLV": "CLE",

    # Houston
    "HST": "HOU",

    # Jacksonville
    "JAC": "JAX",

    # Green Bay
    "GNB": "GB",

    # Kansas City
    "KAN": "KC",

    # Los Angeles Rams / St. Louis Rams
    "STL": "LA",
    "SL": "LA",
    "LAR": "LA",

    # Los Angeles Chargers / San Diego Chargers
    "SD": "LAC",
    "SDG": "LAC",

    # Las Vegas / Oakland Raiders
    "OAK": "LV",
    "LVR": "LV",

    # New England
    "NWE": "NE",

    # New Orleans
    "NOR": "NO",

    # San Francisco
    "SFO": "SF",

    # Tampa Bay
    "TAM": "TB",

    # Washington
    "WSH": "WAS",
    "WFT": "WAS"
}

def standardize_team_column(df, column_name):
    if column_name not in df.columns:
        return df

    return df.with_columns(
        pl.col(column_name)
        .replace(TEAM_MAP)
        .alias(column_name)
    )

## Apply Team Standardization

The canonical team mapping is now applied across the raw datasets.

Only fields representing NFL team identities are standardized. Original raw files remain unchanged on disk.

In [73]:
# Schedules
schedules = standardize_team_column(schedules, "home_team")
schedules = standardize_team_column(schedules, "away_team")

# Rosters
rosters = standardize_team_column(rosters, "team")

# Player statistics
player_stats = standardize_team_column(player_stats, "recent_team")

# Participation
participation = standardize_team_column(participation, "possession_team")

# Snap counts
snap_counts = standardize_team_column(snap_counts, "team")
snap_counts = standardize_team_column(snap_counts, "opponent")

# Depth charts
depth_charts = standardize_team_column(depth_charts, "team")
depth_charts = standardize_team_column(depth_charts, "club_code")

# Draft picks
draft_picks = standardize_team_column(draft_picks, "team")

# Injuries
injuries = standardize_team_column(injuries, "team")

# Trades
trades = standardize_team_column(trades, "gave")
trades = standardize_team_column(trades, "received")

# Weekly rosters
weekly_rosters = standardize_team_column(weekly_rosters, "team")

print("Team abbreviations standardized.")

Team abbreviations standardized.


## Validate Team Identifiers

After standardization, team identifiers are reviewed across datasets to identify any unexpected abbreviations, historical aliases, or source specific naming conventions that still require attention.

In [74]:
team_checks = {
    "schedules_home": schedules.select("home_team").unique().sort("home_team"),
    "schedules_away": schedules.select("away_team").unique().sort("away_team"),
    "rosters": rosters.select("team").unique().sort("team"),
    "player_stats": player_stats.select("recent_team").unique().sort("recent_team"),
    "participation": participation.select("possession_team").unique().sort("possession_team"),
    "snap_counts_team": snap_counts.select("team").unique().sort("team"),
    "snap_counts_opponent": snap_counts.select("opponent").unique().sort("opponent"),
    "draft_picks": draft_picks.select("team").unique().sort("team"),
    "injuries": injuries.select("team").unique().sort("team"),
    "weekly_rosters": weekly_rosters.select("team").unique().sort("team"),
}

for name, values in team_checks.items():
    print(f"\n{name}")
    print(values.to_series().to_list())


schedules_home
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

schedules_away
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

rosters
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

player_stats
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

participation
[None, '', 'ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'C

# Standardize Season and Game Type

The projection model is primarily trained on regular season NFL performance.

Before filtering any dataset, game type fields are inspected to understand how regular season, postseason, and preseason records are represented across sources.

In [75]:
game_type_checks = {}

for name, df in {
    "schedules": schedules,
    "player_stats": player_stats,
    "participation": participation,
    "snap_counts": snap_counts,
    "injuries": injuries,
    "weekly_rosters": weekly_rosters,
}.items():

    relevant_columns = [
        col for col in ["game_type", "season_type"]
        if col in df.columns
    ]

    for col in relevant_columns:
        values = (
            df.select(col)
            .unique()
            .sort(col)
            .to_series()
            .to_list()
        )

        game_type_checks[f"{name}.{col}"] = values

for name, values in game_type_checks.items():
    print(f"{name}: {values}")

schedules.game_type: ['CON', 'DIV', 'REG', 'SB', 'WC']
player_stats.season_type: ['REG']
snap_counts.game_type: ['CON', 'DIV', 'REG', 'SB', 'WC']
injuries.game_type: ['CON', 'DIV', 'REG', 'SB', 'WC']
injuries.season_type: [None, 'POST', 'REG']
weekly_rosters.game_type: ['CON', 'DIV', 'REG', 'SB', 'WC']


## Create Regular Season Datasets

The primary projection features will be based on regular season performance so that teams are evaluated over comparable portions of each season.

Postseason records remain preserved in the raw datasets and can still be used later for separate analysis if needed.

In [76]:
schedules_reg = schedules.filter(
    pl.col("game_type") == "REG"
)

player_stats_reg = player_stats.filter(
    pl.col("season_type") == "REG"
)

snap_counts_reg = snap_counts.filter(
    pl.col("game_type") == "REG"
)

injuries_reg = injuries.filter(
    pl.col("game_type") == "REG"
)

weekly_rosters_reg = weekly_rosters.filter(
    pl.col("game_type") == "REG"
)

print("Regular-season datasets created.")
print()
print(f"Schedules:       {schedules.height:,} -> {schedules_reg.height:,}")
print(f"Player Stats:    {player_stats.height:,} -> {player_stats_reg.height:,}")
print(f"Snap Counts:     {snap_counts.height:,} -> {snap_counts_reg.height:,}")
print(f"Injuries:        {injuries.height:,} -> {injuries_reg.height:,}")
print(f"Weekly Rosters:  {weekly_rosters.height:,} -> {weekly_rosters_reg.height:,}")

Regular-season datasets created.

Schedules:       3,028 -> 2,895
Player Stats:    21,377 -> 21,377
Snap Counts:     276,948 -> 264,774
Injuries:        60,788 -> 58,449
Weekly Rosters:  498,381 -> 475,749


## Validate Regular Season Schedule Coverage

Regular season game counts are checked by season to confirm that the schedule dataset has complete historical coverage.

The NFL expanded from a 16 game schedule to a 17 game schedule beginning in 2021, so expected league wide game totals differ across the modeling period.

In [77]:
schedule_counts = (
    schedules_reg
    .group_by("season")
    .agg(
        pl.len().alias("games"),
        pl.col("home_team").n_unique().alias("home_teams"),
        pl.col("away_team").n_unique().alias("away_teams"),
        pl.col("week").min().alias("min_week"),
        pl.col("week").max().alias("max_week")
    )
    .sort("season")
)

schedule_counts

season,games,home_teams,away_teams,min_week,max_week
i32,u32,u32,u32,i32,i32
2015,256,32,32,1,17
2016,256,32,32,1,17
2017,256,32,32,1,17
2018,256,32,32,1,17
2019,256,32,32,1,17
…,…,…,…,…,…
2021,272,32,32,1,18
2022,271,32,32,1,18
2023,272,32,32,1,18


In [78]:
EXPECTED_REGULAR_SEASON_GAMES = {
    2015: 256,
    2016: 256,
    2017: 256,
    2018: 256,
    2019: 256,
    2020: 256,
    2021: 272,
    2022: 271,  # BUF-CIN Week 17 game was canceled
    2023: 272,
    2024: 272,
    2025: 272,
}

schedule_validation = (
    schedule_counts
    .with_columns(
        pl.col("season")
        .replace_strict(
            EXPECTED_REGULAR_SEASON_GAMES,
            default=None
        )
        .alias("expected_games")
    )
    .with_columns(
        (pl.col("games") == pl.col("expected_games"))
        .alias("game_count_valid"),

        (
            (pl.col("home_teams") == 32) &
            (pl.col("away_teams") == 32)
        ).alias("team_count_valid")
    )
)

schedule_validation

season,games,home_teams,away_teams,min_week,max_week,expected_games,game_count_valid,team_count_valid
i32,u32,u32,u32,i32,i32,i64,bool,bool
2015,256,32,32,1,17,256,true,true
2016,256,32,32,1,17,256,true,true
2017,256,32,32,1,17,256,true,true
2018,256,32,32,1,17,256,true,true
2019,256,32,32,1,17,256,true,true
…,…,…,…,…,…,…,…,…
2021,272,32,32,1,18,272,true,true
2022,271,32,32,1,18,271,true,true
2023,272,32,32,1,18,272,true,true


In [79]:
assert schedule_validation["game_count_valid"].all()
assert schedule_validation["team_count_valid"].all()

print("Regular season schedule validation passed.")

Regular season schedule validation passed.


# Clean Schedule Data

The regular season schedule is reduced to a standardized game level table.

This table will serve as the canonical game reference for downstream joins and will preserve game identity, teams, scores, location, rest, and schedule context.

In [80]:
schedule_clean = (
    schedules_reg
    .select([
        "game_id",
        "season",
        "week",
        "gameday",
        "weekday",
        "gametime",
        "away_team",
        "away_score",
        "home_team",
        "home_score",
        "location",
        "result",
        "total",
        "overtime",
        "away_rest",
        "home_rest",
    ])
    .sort(["season", "week", "gameday", "game_id"])
)

print(schedule_clean.shape)
schedule_clean.head(10)

(2895, 16)


game_id,season,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,away_rest,home_rest
str,i32,i32,str,str,str,str,i32,str,i32,str,i32,i32,i32,i32,i32
"""2015_01_PIT_NE""",2015,1,"""2015-09-10""","""Thursday""","""20:30""","""PIT""",21,"""NE""",28,"""Home""",7,49,0,7,7
"""2015_01_BAL_DEN""",2015,1,"""2015-09-13""","""Sunday""","""16:25""","""BAL""",13,"""DEN""",19,"""Home""",6,32,0,7,7
"""2015_01_CAR_JAX""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""CAR""",20,"""JAX""",9,"""Home""",-11,29,0,7,7
"""2015_01_CIN_OAK""",2015,1,"""2015-09-13""","""Sunday""","""16:25""","""CIN""",33,"""LV""",13,"""Home""",-20,46,0,7,7
"""2015_01_CLE_NYJ""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""CLE""",10,"""NYJ""",31,"""Home""",21,41,0,7,7
"""2015_01_DET_SD""",2015,1,"""2015-09-13""","""Sunday""","""16:05""","""DET""",28,"""LAC""",33,"""Home""",5,61,0,7,7
"""2015_01_GB_CHI""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""GB""",31,"""CHI""",23,"""Home""",-8,54,0,7,7
"""2015_01_IND_BUF""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""IND""",14,"""BUF""",27,"""Home""",13,41,0,7,7
"""2015_01_KC_HOU""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""KC""",27,"""HOU""",20,"""Home""",-7,47,0,7,7


In [81]:
duplicate_game_ids = (
    schedule_clean
    .group_by("game_id")
    .len()
    .filter(pl.col("len") > 1)
)

print(f"Duplicate game IDs: {duplicate_game_ids.height}")
print(f"Null game IDs: {schedule_clean['game_id'].null_count()}")

Duplicate game IDs: 0
Null game IDs: 0


## Schedule Missing Value Audit

The cleaned schedule is checked for missing values before being saved.

Missing values are reviewed rather than automatically filled so that source specific issues are not hidden during cleaning.

In [82]:
schedule_nulls = pl.DataFrame({
    "column": schedule_clean.columns,
    "null_count": [
        schedule_clean[col].null_count()
        for col in schedule_clean.columns
    ]
}).filter(
    pl.col("null_count") > 0
)

schedule_nulls

column,null_count
str,i64


In [83]:
schedule_clean_output = PROCESSED_DIR / "schedule_clean.parquet"

schedule_clean.write_parquet(schedule_clean_output)

print(f"Saved {schedule_clean.height:,} cleaned games to:")
print(schedule_clean_output)

Saved 2,895 cleaned games to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\schedule_clean.parquet


# Clean Season Roster Data

Season roster data provides the foundation for identifying players, positions, experience, and team membership across seasons.

The cleaning process preserves stable player identifiers wherever possible and standardizes the fields that will later connect roster information with performance, snap count, injury, and personnel datasets.

In [84]:
roster_id_check = pl.DataFrame({
    "identifier": [
        "gsis_id",
        "espn_id",
        "pfr_id",
        "sportradar_id"
    ],
    "non_null": [
        rosters["gsis_id"].is_not_null().sum(),
        rosters["espn_id"].is_not_null().sum(),
        rosters["pfr_id"].is_not_null().sum(),
        rosters["sportradar_id"].is_not_null().sum(),
    ],
    "null": [
        rosters["gsis_id"].null_count(),
        rosters["espn_id"].null_count(),
        rosters["pfr_id"].null_count(),
        rosters["sportradar_id"].null_count(),
    ]
})

roster_id_check

identifier,non_null,null
str,i64,i64
"""gsis_id""",33184,11
"""espn_id""",21142,12053
"""pfr_id""",16126,17069
"""sportradar_id""",21838,11357


## Player Identity Rules

`gsis_id` is used as the primary player identifier throughout the project because it provides nearly complete coverage and is shared across multiple NFL datasets.

Players without a GSIS ID are retained rather than removed or assigned an inferred identifier. These records may remain unmatched in downstream player level joins.

Player names are preserved for display and validation purposes but are not treated as unique identifiers.

In [85]:
rosters_clean = (
    rosters
    .select([
        "season",
        "team",
        "gsis_id",
        "full_name",
        "football_name",
        "position",
        "depth_chart_position",
        "status",
        "years_exp",
        "birth_date",
        "height",
        "weight",
        "college",
        "espn_id",
        "pfr_id",
        "sportradar_id",
    ])
    .sort([
        "season",
        "team",
        "gsis_id"
    ])
)

print(rosters_clean.shape)
rosters_clean.head(10)

(33195, 16)


season,team,gsis_id,full_name,football_name,position,depth_chart_position,status,years_exp,birth_date,height,weight,college,espn_id,pfr_id,sportradar_id
i32,str,str,str,str,str,str,str,i32,date,f64,i32,str,str,str,str
2015,"""ARI""","""00-0019435""","""Mike Leach""","""Mike""","""LS""",null,"""ACT""",15,1976-10-18,74.0,235,"""William & Mary""",null,null,null
2015,"""ARI""","""00-0021146""","""Dwight Freeney""","""Dwight""","""OLB""",null,"""ACT""",13,1980-02-19,73.0,268,"""Syracuse""",null,null,null
2015,"""ARI""","""00-0021429""","""Carson Palmer""","""Carson""","""QB""",null,"""ACT""",12,1979-12-27,77.0,235,"""USC""","""4459""","""PalmCa00""","""57ad34b3-f60d-4b2d-9e01-3cb5a9…"
2015,"""ARI""","""00-0021998""","""Cory Redding""","""Cory""","""DT""",null,"""RES""",12,1980-11-15,76.0,318,"""Texas""",null,null,null
2015,"""ARI""","""00-0022695""","""Jason Babin""","""Jason""","""OLB""",null,"""ACT""",11,1980-05-24,75.0,267,"""Western Michigan""",null,null,null
2015,"""ARI""","""00-0022921""","""Larry Fitzgerald""","""Larry""","""WR""",null,"""ACT""",11,1983-08-31,75.0,218,"""Pittsburgh""","""5528""","""FitzLa00""","""b6a61b38-5cfa-46eb-b1c5-b0255d…"
2015,"""ARI""","""00-0024306""","""Frostee Rucker""","""Frostee""","""DT""",null,"""ACT""",9,1983-09-14,75.0,280,"""USC""","""9677""","""RuckFr99""","""36648f14-5fe5-40f3-ade1-ef53c8…"
2015,"""ARI""","""00-0025430""","""Drew Stanton""","""Drew""","""QB""",null,"""ACT""",8,1984-05-07,75.0,243,"""Michigan State""","""10487""","""StanDr00""","""22fb2b54-4936-4e8a-a48d-62096c…"
2015,"""ARI""","""00-0025433""","""LaMarr Woodley""","""LaMarr""","""LB""",null,"""RES""",8,1984-11-03,74.0,265,"""Michigan""",null,null,null


In [86]:
roster_duplicates = (
    rosters_clean
    .filter(pl.col("gsis_id").is_not_null())
    .group_by([
        "season",
        "team",
        "gsis_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(f"Duplicate player-season-team combinations: {roster_duplicates.height}")

roster_duplicates.head(20)

Duplicate player-season-team combinations: 0


season,team,gsis_id,len
i32,str,str,u32


## Validate Season Roster Coverage

Season roster coverage is checked by season to confirm that all 32 NFL teams are represented before the cleaned dataset is saved.

In [87]:
roster_coverage = (
    rosters_clean
    .group_by("season")
    .agg([
        pl.len().alias("roster_records"),
        pl.col("team").n_unique().alias("teams"),
        pl.col("gsis_id").null_count().alias("missing_gsis_ids"),
    ])
    .sort("season")
)

roster_coverage

season,roster_records,teams,missing_gsis_ids
i32,u32,u32,u32
2015,2190,32,1
2016,3061,32,0
2017,3082,32,0
2018,3142,32,1
2019,3114,32,1
…,…,…,…
2021,2961,32,1
2022,3134,32,1
2023,3090,32,1


In [88]:
rosters_clean_output = PROCESSED_DIR / "rosters_clean.parquet"

rosters_clean.write_parquet(rosters_clean_output)

print(f"Saved {rosters_clean.height:,} cleaned roster records to:")
print(rosters_clean_output)

Saved 33,195 cleaned roster records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\rosters_clean.parquet


# Clean Player Statistics

Player statistics provide the individual performance data used later for quarterback projections, skill position evaluation, and position group analysis.

The cleaning process focuses on preserving player identity, validating season coverage, identifying duplicate player season records, and retaining the statistical fields needed for downstream modeling.

In [89]:
player_stats_id_check = pl.DataFrame({
    "field": [
        "player_id",
        "player_name",
        "player_display_name",
        "recent_team"
    ],
    "null_count": [
        player_stats_reg["player_id"].null_count(),
        player_stats_reg["player_name"].null_count(),
        player_stats_reg["player_display_name"].null_count(),
        player_stats_reg["recent_team"].null_count(),
    ]
})

player_stats_id_check

field,null_count
str,i64
"""player_id""",11
"""player_name""",6
"""player_display_name""",11
"""recent_team""",0


In [90]:
player_stats_duplicates = (
    player_stats_reg
    .filter(pl.col("player_id").is_not_null())
    .group_by([
        "season",
        "player_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(
    f"Duplicate player-season combinations: "
    f"{player_stats_duplicates.height}"
)

player_stats_duplicates.head(20)

Duplicate player-season combinations: 0


season,player_id,len
i32,str,u32


## Remove Non-Player Aggregate Records

A small number of records without player IDs represent league or team level aggregate rows rather than individual players.

These records contain season-wide game counts and no meaningful player level usage. Because the player statistics dataset is intended to contain one record per individual player and season, these aggregate records are excluded from the cleaned dataset.

In [91]:
player_stats_clean = (
    player_stats_reg
    .filter(pl.col("player_id").is_not_null())
)

print(f"Original records: {player_stats_reg.height:,}")
print(f"Removed non-player records: {player_stats_reg.height - player_stats_clean.height:,}")
print(f"Clean player records: {player_stats_clean.height:,}")

Original records: 21,377
Removed non-player records: 11
Clean player records: 21,366


In [92]:
print(f"Missing player IDs: {player_stats_clean['player_id'].null_count()}")
print(f"Missing display names: {player_stats_clean['player_display_name'].null_count()}")

player_stats_clean.group_by("season").agg(
    pl.len().alias("players")
).sort("season")

Missing player IDs: 0
Missing display names: 0


season,players
i32,u32
2015,1845
2016,1855
2017,1868
2018,1883
2019,1888
…,…
2021,2081
2022,2006
2023,1942


In [93]:
player_stats_clean_output = PROCESSED_DIR / "player_stats_clean.parquet"

player_stats_clean.write_parquet(player_stats_clean_output)

print(f"Saved {player_stats_clean.height:,} cleaned player stat records to:")
print(player_stats_clean_output)

Saved 21,366 cleaned player stat records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\player_stats_clean.parquet


# Clean Snap Count Data

Snap counts provide player level playing time at the game level.

This dataset will later support returning production, starter continuity, offensive line stability, defensive front rotations, and the importance of roster additions and departures.

The cleaning process focuses on player identity, game identity, duplicate records, and missing snap information.

In [94]:
snap_id_check = pl.DataFrame({
    "field": [
        "game_id",
        "pfr_player_id",
        "player",
        "team",
        "opponent"
    ],
    "null_count": [
        snap_counts_reg["game_id"].null_count(),
        snap_counts_reg["pfr_player_id"].null_count(),
        snap_counts_reg["player"].null_count(),
        snap_counts_reg["team"].null_count(),
        snap_counts_reg["opponent"].null_count(),
    ]
})

snap_id_check

field,null_count
str,i64
"""game_id""",0
"""pfr_player_id""",0
"""player""",0
"""team""",0
"""opponent""",0


In [95]:
snap_duplicates = (
    snap_counts_reg
    .group_by([
        "game_id",
        "team",
        "pfr_player_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(f"Duplicate player-game combinations: {snap_duplicates.height}")

snap_duplicates.head(20)

Duplicate player-game combinations: 0


game_id,team,pfr_player_id,len
str,str,str,u32


In [96]:
snap_counts_clean = (
    snap_counts_reg
    .select([
        "game_id",
        "season",
        "week",
        "player",
        "pfr_player_id",
        "position",
        "team",
        "opponent",
        "offense_snaps",
        "offense_pct",
        "defense_snaps",
        "defense_pct",
        "st_snaps",
        "st_pct",
    ])
    .sort([
        "season",
        "week",
        "game_id",
        "team",
        "pfr_player_id"
    ])
)

print(snap_counts_clean.shape)
snap_counts_clean.head(10)

(264774, 14)


game_id,season,week,player,pfr_player_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
str,i32,i32,str,str,str,str,str,f64,f64,f64,f64,f64,f64
"""2015_01_BAL_DEN""",2015,1,"""Kamar Aiken""","""AikeKa00""","""WR""","""BAL""","""DEN""",44.0,0.76,0.0,0.0,1.0,0.03
"""2015_01_BAL_DEN""",2015,1,"""Javorius Allen""","""AlleJa01""","""RB""","""BAL""","""DEN""",13.0,0.22,0.0,0.0,6.0,0.21
"""2015_01_BAL_DEN""",2015,1,"""Kyle Arrington""","""ArriKy00""","""CB""","""BAL""","""DEN""",0.0,0.0,39.0,0.56,0.0,0.0
"""2015_01_BAL_DEN""",2015,1,"""Nick Boyle""","""BoylNi00""","""TE""","""BAL""","""DEN""",8.0,0.14,0.0,0.0,11.0,0.38
"""2015_01_BAL_DEN""",2015,1,"""Terrence Brooks""","""BrooTe00""","""FS""","""BAL""","""DEN""",0.0,0.0,0.0,0.0,19.0,0.66
"""2015_01_BAL_DEN""",2015,1,"""Arthur Brown""","""BrowAr00""","""LB""","""BAL""","""DEN""",0.0,0.0,0.0,0.0,14.0,0.48
"""2015_01_BAL_DEN""",2015,1,"""Marlon Brown""","""BrowMa00""","""WR""","""BAL""","""DEN""",34.0,0.59,0.0,0.0,3.0,0.1
"""2015_01_BAL_DEN""",2015,1,"""Michael Campanaro""","""CampMi02""","""WR""","""BAL""","""DEN""",11.0,0.19,0.0,0.0,6.0,0.21
"""2015_01_BAL_DEN""",2015,1,"""Chris Canty""","""CantCh21""","""DE""","""BAL""","""DEN""",0.0,0.0,42.0,0.6,5.0,0.17


In [97]:
snap_value_check = pl.DataFrame({
    "field": [
        "offense_snaps",
        "offense_pct",
        "defense_snaps",
        "defense_pct",
        "st_snaps",
        "st_pct"
    ],
    "null_count": [
        snap_counts_clean[col].null_count()
        for col in [
            "offense_snaps",
            "offense_pct",
            "defense_snaps",
            "defense_pct",
            "st_snaps",
            "st_pct"
        ]
    ]
})

snap_value_check

field,null_count
str,i64
"""offense_snaps""",0
"""offense_pct""",0
"""defense_snaps""",0
"""defense_pct""",0
"""st_snaps""",0
"""st_pct""",0


In [98]:
snap_pct_validation = pl.DataFrame({
    "field": [
        "offense_pct",
        "defense_pct",
        "st_pct"
    ],
    "min_value": [
        snap_counts_clean["offense_pct"].min(),
        snap_counts_clean["defense_pct"].min(),
        snap_counts_clean["st_pct"].min(),
    ],
    "max_value": [
        snap_counts_clean["offense_pct"].max(),
        snap_counts_clean["defense_pct"].max(),
        snap_counts_clean["st_pct"].max(),
    ]
})

snap_pct_validation

field,min_value,max_value
str,f64,f64
"""offense_pct""",0.0,1.0
"""defense_pct""",0.0,1.0
"""st_pct""",0.0,1.01


In [99]:
assert (
    snap_counts_clean["offense_pct"].min() >= 0
    and snap_counts_clean["offense_pct"].max() <= 1
)

assert (
    snap_counts_clean["defense_pct"].min() >= 0
    and snap_counts_clean["defense_pct"].max() <= 1
)

assert (
    snap_counts_clean["st_pct"].min() >= 0
    and snap_counts_clean["st_pct"].max() <= 1.01
)

print("Snap percentage validation passed.")

Snap percentage validation passed.


In [100]:
snap_counts_clean_output = PROCESSED_DIR / "snap_counts_clean.parquet"

snap_counts_clean.write_parquet(snap_counts_clean_output)

print(f"Saved {snap_counts_clean.height:,} cleaned snap count records to:")
print(snap_counts_clean_output)

Saved 264,774 cleaned snap count records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\snap_counts_clean.parquet


# Clean Participation and Personnel Data

Participation data provides play level information about offensive and defensive personnel, formations, box counts, pass rushers, pressure, and time to throw.

These fields will later support team level features describing offensive structure, pass protection, defensive pressure, front usage, and personnel tendencies.

Because participation data begins in 2016, features derived from this source will have a shorter historical window than the core play-by-play and schedule datasets.

In [101]:
participation_id_check = pl.DataFrame({
    "field": [
        "nflverse_game_id",
        "play_id",
        "possession_team"
    ],
    "null_count": [
        participation["nflverse_game_id"].null_count(),
        participation["play_id"].null_count(),
        participation["possession_team"].null_count(),
    ]
})

participation_id_check

field,null_count
str,i64
"""nflverse_game_id""",0
"""play_id""",0
"""possession_team""",4


In [102]:
participation_duplicates = (
    participation
    .group_by([
        "nflverse_game_id",
        "play_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(
    f"Duplicate game-play combinations: "
    f"{participation_duplicates.height}"
)

participation_duplicates.head(20)

Duplicate game-play combinations: 0


nflverse_game_id,play_id,len
str,f64,u32


## Restrict Participation Data to Regular Season Games

The participation dataset does not contain a dedicated season type field.

Regular season participation records are therefore identified by matching `nflverse_game_id` against the validated regular season game IDs contained in the cleaned schedule dataset. This ensures postseason and other non-regular season records are excluded consistently.

In [103]:
regular_season_game_ids = (
    schedule_clean
    .select(
        pl.col("game_id").alias("nflverse_game_id")
    )
    .unique()
)

participation_reg = (
    participation
    .join(
        regular_season_game_ids,
        on="nflverse_game_id",
        how="inner"
    )
)

print(f"Original participation records: {participation.height:,}")
print(f"Regular season participation records: {participation_reg.height:,}")
print(f"Removed records: {participation.height - participation_reg.height:,}")

Original participation records: 478,989
Regular season participation records: 457,830
Removed records: 21,159


In [104]:
participation_reg = participation_reg.with_columns(
    pl.col("nflverse_game_id")
    .str.slice(0, 4)
    .cast(pl.Int32)
    .alias("season")
)

In [105]:
participation_season_check = (
    participation_reg
    .group_by("season")
    .agg([
        pl.len().alias("plays"),
        pl.col("nflverse_game_id").n_unique().alias("games"),
        pl.col("possession_team").n_unique().alias("teams"),
    ])
    .sort("season")
)

participation_season_check

season,plays,games,teams
i32,u32,u32,u32
2016,46468,256,33
2017,46000,256,33
2018,45850,256,33
2019,46091,256,33
2020,46189,256,33
2021,48416,272,34
2022,47844,271,34
2023,44057,272,32
2024,43866,272,32


In [106]:
VALID_TEAMS = [
    "ARI", "ATL", "BAL", "BUF", "CAR", "CHI", "CIN", "CLE",
    "DAL", "DEN", "DET", "GB", "HOU", "IND", "JAX", "KC",
    "LA", "LAC", "LV", "MIA", "MIN", "NE", "NO", "NYG",
    "NYJ", "PHI", "PIT", "SEA", "SF", "TB", "TEN", "WAS"
]

participation_bad_team_codes = (
    participation_reg
    .filter(
        pl.col("possession_team").is_not_null() &
        (pl.col("possession_team") != "") &
        (~pl.col("possession_team").is_in(VALID_TEAMS))
    )
    .group_by([
        "season",
        "possession_team"
    ])
    .len()
    .sort([
        "season",
        "possession_team"
    ])
)

participation_bad_team_codes

season,possession_team,len
i32,str,u32


## Remove Unusable Participation Records

Participation records without an identifiable possession team are excluded from the cleaned dataset.

These records contain almost no usable formation, pressure, pass rush, or box count information and cannot be reliably assigned to an offensive team. A small number contain personnel strings, but without a possession team they cannot support team level feature engineering.

In [107]:
participation_clean = (
    participation_reg
    .filter(
        pl.col("possession_team").is_not_null() &
        (pl.col("possession_team") != "")
    )
)

print(f"Regular season records: {participation_reg.height:,}")
print(f"Removed unusable records: {participation_reg.height - participation_clean.height:,}")
print(f"Clean participation records: {participation_clean.height:,}")

Regular season records: 457,830
Removed unusable records: 27,863
Clean participation records: 429,967


In [108]:
participation_field_coverage = pl.DataFrame({
    "field": [
        "offense_formation",
        "offense_personnel",
        "defenders_in_box",
        "defense_personnel",
        "number_of_pass_rushers",
        "time_to_throw",
        "was_pressure"
    ],
    "non_null": [
        participation_clean[col].is_not_null().sum()
        for col in [
            "offense_formation",
            "offense_personnel",
            "defenders_in_box",
            "defense_personnel",
            "number_of_pass_rushers",
            "time_to_throw",
            "was_pressure"
        ]
    ]
}).with_columns(
    (
        pl.col("non_null") / participation_clean.height
    ).alias("coverage_pct")
)

participation_field_coverage

field,non_null,coverage_pct
str,i64,f64
"""offense_formation""",342369,0.796268
"""offense_personnel""",379398,0.882389
"""defenders_in_box""",372990,0.867485
"""defense_personnel""",379398,0.882389
"""number_of_pass_rushers""",268845,0.625269
"""time_to_throw""",182365,0.424137
"""was_pressure""",256628,0.596855


## Preserve Participation Feature Coverage

Missing participation values are retained as null rather than replaced with zero because many fields apply only to specific play types or situations.

The cleaned participation dataset preserves personnel, formation, pressure, timing, and player participation information so feature selection and aggregation can be performed later during feature engineering.

In [109]:
participation_clean = (
    participation_clean
    .select([
        "season",
        "nflverse_game_id",
        "play_id",
        "possession_team",
        "offense_formation",
        "offense_personnel",
        "defenders_in_box",
        "defense_personnel",
        "number_of_pass_rushers",
        "players_on_play",
        "offense_players",
        "defense_players",
        "n_offense",
        "n_defense",
        "ngs_air_yards",
        "time_to_throw",
        "was_pressure",
        "route",
        "defense_man_zone_type",
        "defense_coverage_type",
        "offense_names",
        "defense_names",
        "offense_positions",
        "defense_positions",
        "offense_numbers",
        "defense_numbers",
    ])
    .sort([
        "season",
        "nflverse_game_id",
        "play_id"
    ])
)

print(participation_clean.shape)
participation_clean.head(10)

(429967, 26)


season,nflverse_game_id,play_id,possession_team,offense_formation,offense_personnel,defenders_in_box,defense_personnel,number_of_pass_rushers,players_on_play,offense_players,defense_players,n_offense,n_defense,ngs_air_yards,time_to_throw,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
i32,str,f64,str,str,str,i32,str,i32,str,str,str,i32,i32,f64,f64,bool,str,str,str,str,str,str,str,str,str
2016,"""2016_01_BUF_BAL""",36.0,"""BUF""",null,null,null,null,null,"""40078;40151;37977;36060;40494;…","""00-0030041;00-0030073;00-00297…","""00-0029892;00-0029895;00-00284…",10,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",58.0,"""BAL""","""I_FORM""","""2 RB, 1 TE, 2 WR""",8,"""4 DL, 3 LB, 4 DB""",null,"""38540;41302;40078;35553;38582;…","""00-0029892;00-0027714;00-00329…","""00-0029542;00-0031171;00-00295…",11,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",85.0,"""BAL""","""SHOTGUN""","""2 RB, 1 TE, 2 WR""",6,"""4 DL, 3 LB, 4 DB""",3,"""41302;40078;38540;35553;43295;…","""00-0029892;00-0027714;00-00329…","""00-0031171;00-0029542;00-00295…",11,11,-4.53,2.404,false,"""FLAT""",null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",109.0,"""BAL""","""SHOTGUN""","""2 RB, 1 TE, 2 WR""",7,"""4 DL, 3 LB, 4 DB""",null,"""40078;38540;41302;35553;43295;…","""00-0029892;00-0027714;00-00329…","""00-0029542;00-0031171;00-00295…",11,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",130.0,"""BAL""","""I_FORM""","""2 RB, 1 TE, 2 WR""",7,"""4 DL, 3 LB, 4 DB""",5,"""40078;38540;41302;35553;38582;…","""00-0029892;00-0027714;00-00329…","""00-0029542;00-0031171;00-00295…",11,11,1.71,2.093,false,"""FLAT""",null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",154.0,"""BAL""","""I_FORM""","""1 RB, 2 TE, 2 WR""",8,"""4 DL, 3 LB, 4 DB""",null,"""41302;38540;43295;38582;40053;…","""00-0032965;00-0029893;00-00262…","""00-0031171;00-0029542;00-00295…",11,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",175.0,"""BAL""",null,"""1 RB, 2 TE, 2 WR""",null,"""4 DL, 3 LB, 4 DB""",null,"""38540;41302;43295;38582;40053;…","""00-0032965;00-0029893;00-00262…","""00-0029542;00-0031171;00-00295…",11,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",198.0,"""BAL""",null,null,null,null,null,"""40078;35575;36060;37977;40494;…","""00-0029892;00-0027736;00-00275…","""00-0030041;00-0030073;00-00304…",10,10,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",216.0,"""BUF""","""SHOTGUN""","""1 RB, 2 TE, 2 WR""",7,"""3 DL, 4 LB, 4 DB""",4,"""34479;40111;41277;37249;41230;…","""00-0027004;00-0030046;00-00281…","""00-0031170;00-0027560;00-00279…",11,10,1.61,2.537,false,"""ANGLE""",null,null,null,null,null,null,null,null


In [110]:
assert participation_clean.select(
    pl.struct(["nflverse_game_id", "play_id"]).n_unique()
).item() == participation_clean.height

assert (
    participation_clean
    .filter(
        pl.col("possession_team").is_null() |
        (pl.col("possession_team") == "")
    )
    .height == 0
)

assert (
    participation_clean
    .filter(~pl.col("possession_team").is_in(VALID_TEAMS))
    .height == 0
)

print("Participation validation passed.")

Participation validation passed.


In [111]:
participation_clean_output = PROCESSED_DIR / "participation_clean.parquet"

participation_clean.write_parquet(participation_clean_output)

print(f"Saved {participation_clean.height:,} cleaned participation records to:")
print(participation_clean_output)

Saved 429,967 cleaned participation records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\participation_clean.parquet


# Clean Injury Data

Injury reports provide weekly information about player availability, reported injuries, game status, and practice participation.

These records will later be combined with roster, snap count, and player information to create team level availability features. This allows the model to distinguish between injuries to high usage players and injuries to players with limited roles.

The raw injury data is first restricted to the regular season and standardized before any injury impact features are created.

In [112]:
injury_key_check = (
    injuries_reg
    .group_by([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(f"Duplicate season-week-team-player combinations: {injury_key_check.height}")

injury_key_check.head(20)

Duplicate season-week-team-player combinations: 2


season,week,team,gsis_id,len
f64,f64,str,str,u32
2024.0,15.0,"""NYJ""","""00-0034270""",2
2024.0,15.0,"""HOU""","""00-0039359""",2


In [113]:
injury_identifier_check = pl.DataFrame({
    "field": [
        "gsis_id",
        "full_name",
        "team",
        "week"
    ],
    "null_count": [
        injuries_reg["gsis_id"].null_count(),
        injuries_reg["full_name"].null_count(),
        injuries_reg["team"].null_count(),
        injuries_reg["week"].null_count()
    ]
})

injury_identifier_check

field,null_count
str,i64
"""gsis_id""",0
"""full_name""",0
"""team""",0
"""week""",0


In [114]:
injuries_clean = (
    injuries_reg
    .sort("date_modified")
    .unique(
        subset=[
            "season",
            "week",
            "team",
            "gsis_id"
        ],
        keep="last"
    )
    .sort([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
)

print(f"Original regular season injury records: {injuries_reg.height:,}")
print(f"Clean injury records: {injuries_clean.height:,}")
print(f"Updated records removed: {injuries_reg.height - injuries_clean.height:,}")

Original regular season injury records: 58,449
Clean injury records: 58,447
Updated records removed: 2


## Standardize Injury Status Fields

Blank injury and practice status strings are converted to null values so missing information is represented consistently.

Original injury terminology is otherwise preserved. Status categories are not collapsed or reinterpreted during cleaning because game designations and practice participation provide different types of information.

In [115]:
injuries_clean = injuries_clean.with_columns([
    pl.when(
        pl.col("report_status").str.strip_chars() == ""
    )
    .then(None)
    .otherwise(pl.col("report_status"))
    .alias("report_status"),

    pl.when(
        pl.col("practice_status").str.strip_chars() == ""
    )
    .then(None)
    .otherwise(pl.col("practice_status"))
    .alias("practice_status")
])

In [116]:
injury_text_columns = [
    "report_primary_injury",
    "report_secondary_injury",
    "practice_primary_injury",
    "practice_secondary_injury"
]

injuries_clean = injuries_clean.with_columns([
    pl.when(pl.col(col).str.strip_chars() == "")
    .then(None)
    .otherwise(pl.col(col))
    .alias(col)
    for col in injury_text_columns
])

In [117]:
remaining_injury_duplicates = (
    injuries_clean
    .group_by([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(f"Clean injury records: {injuries_clean.height:,}")
print(f"Remaining duplicates: {remaining_injury_duplicates}")
print(f"Missing GSIS IDs: {injuries_clean['gsis_id'].null_count()}")
print(f"Missing teams: {injuries_clean['team'].null_count()}")

Clean injury records: 58,447
Remaining duplicates: 0
Missing GSIS IDs: 0
Missing teams: 0


In [118]:
injuries_clean_output = PROCESSED_DIR / "injuries_clean.parquet"

injuries_clean.write_parquet(injuries_clean_output)

print(f"Saved {injuries_clean.height:,} cleaned injury records to:")
print(injuries_clean_output)

Saved 58,447 cleaned injury records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\injuries_clean.parquet


# Clean Weekly Roster Data

Weekly roster data gives us a week by week view of team personnel and player status.

I keep the different roster status records because some players can have multiple legitimate transactions or status changes within the same week. This data will later help with roster continuity, player movement, and availability.

In [119]:
weekly_roster_id_check = pl.DataFrame({
    "field": [
        "season",
        "week",
        "team",
        "gsis_id",
        "full_name",
        "status"
    ],
    "null_count": [
        weekly_rosters_reg["season"].null_count(),
        weekly_rosters_reg["week"].null_count(),
        weekly_rosters_reg["team"].null_count(),
        weekly_rosters_reg["gsis_id"].null_count(),
        weekly_rosters_reg["full_name"].null_count(),
        weekly_rosters_reg["status"].null_count(),
    ]
})

weekly_roster_id_check

field,null_count
str,i64
"""season""",0
"""week""",0
"""team""",0
"""gsis_id""",121
"""full_name""",16
"""status""",18


In [120]:
weekly_roster_duplicates = (
    weekly_rosters_reg
    .group_by([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(
    f"Duplicate season-week-team-player combinations: "
    f"{weekly_roster_duplicates.height}"
)

weekly_roster_duplicates.head(20)

Duplicate season-week-team-player combinations: 1702


season,week,team,gsis_id,len
i32,i32,str,str,u32
2015,16,"""SF""","""00-0028409""",4
2015,17,"""CLE""","""00-0031687""",4
2015,17,"""SF""","""00-0028409""",4
2015,5,"""MIA""","""00-0031687""",4
2015,15,"""CLE""","""00-0031687""",4
…,…,…,…,…
2015,13,"""SEA""","""00-0028409""",4
2015,2,"""MIA""","""00-0031687""",4
2015,3,"""MIA""","""00-0031687""",4


In [121]:
weekly_rosters_clean = (
    weekly_rosters_reg
    .select([
        "season",
        "week",
        "team",
        "gsis_id",
        "full_name",
        "football_name",
        "position",
        "depth_chart_position",
        "status",
        "status_description_abbr",
        "years_exp",
        "entry_year",
        "rookie_year",
        "draft_club"
    ])
    .sort([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
)

print(f"Clean weekly roster records: {weekly_rosters_clean.height:,}")
weekly_rosters_clean.head(10)

Clean weekly roster records: 475,749


season,week,team,gsis_id,full_name,football_name,position,depth_chart_position,status,status_description_abbr,years_exp,entry_year,rookie_year,draft_club
i32,i32,str,str,str,str,str,str,str,str,i32,i32,i32,str
2015,1,"""ARI""","""00-0019435""","""Mike Leach""","""Mike""","""LS""",null,"""ACT""","""A01""",15,2000,2000,null
2015,1,"""ARI""","""00-0021429""","""Carson Palmer""","""Carson""","""QB""",null,"""ACT""","""A01""",12,2003,2003,"""CIN"""
2015,1,"""ARI""","""00-0021998""","""Cory Redding""","""Cory""","""DE""",null,"""RES""","""A01""",12,2003,2003,"""DET"""
2015,1,"""ARI""","""00-0022921""","""Larry Fitzgerald""","""Larry""","""WR""",null,"""ACT""","""A01""",11,2004,2004,"""ARZ"""
2015,1,"""ARI""","""00-0024306""","""Frostee Rucker""","""Frostee""","""DT""",null,"""ACT""","""A01""",9,2006,2006,"""CIN"""
2015,1,"""ARI""","""00-0025430""","""Drew Stanton""","""Drew""","""QB""",null,"""ACT""","""A01""",8,2007,2007,"""DET"""
2015,1,"""ARI""","""00-0025433""","""LaMarr Woodley""","""LaMarr""","""LB""",null,"""RES""","""A01""",8,2007,2007,"""PIT"""
2015,1,"""ARI""","""00-0025682""","""Lyle Sendlein""","""Lyle""","""C""",null,"""ACT""","""A01""",8,2007,2007,null
2015,1,"""ARI""","""00-0026164""","""Chris Johnson""","""Chris""","""RB""",null,"""RSR""","""A01""",7,2008,2008,"""TEN"""


In [122]:
weekly_rosters_clean_output = (
    PROCESSED_DIR / "weekly_rosters_clean.parquet"
)

weekly_rosters_clean.write_parquet(
    weekly_rosters_clean_output
)

print(
    f"Saved {weekly_rosters_clean.height:,} "
    f"weekly roster records to:"
)

print(weekly_rosters_clean_output)

Saved 475,749 weekly roster records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\weekly_rosters_clean.parquet


# Clean Draft Data

Draft data gives us each team's rookie additions and the draft capital used on those players.

For now, I just want to standardize the main fields and make sure each draft pick is represented correctly before using it later in offseason features.

In [123]:
draft_picks_clean = (
    draft_picks
    .filter(pl.col("season").is_between(2015, 2025))
    .select([
        "season",
        "round",
        "pick",
        "team",
        "gsis_id",
        "pfr_player_id",
        "pfr_player_name",
        "position",
        "category",
        "side",
        "college",
        "age"
    ])
    .sort([
        "season",
        "pick"
    ])
)

print(draft_picks_clean.shape)
draft_picks_clean.head(10)

(2821, 12)


season,round,pick,team,gsis_id,pfr_player_id,pfr_player_name,position,category,side,college,age
i32,i32,i32,str,str,str,str,str,str,str,str,i32
2015,1,1,"""TB""","""00-0031503""","""WinsJa00""","""Jameis Winston""","""QB""","""QB""","""O""","""Florida St.""",21
2015,1,2,"""TEN""","""00-0032268""","""MariMa01""","""Marcus Mariota""","""QB""","""QB""","""O""","""Oregon""",21
2015,1,3,"""JAX""","""00-0032052""","""FowlDa00""","""Dante Fowler""","""OLB""","""LB""","""D""","""Florida""",21
2015,1,4,"""LV""","""00-0031544""","""CoopAm00""","""Amari Cooper""","""WR""","""WR""","""O""","""Alabama""",21
2015,1,5,"""WAS""","""00-0032053""","""ScheBr00""","""Brandon Scherff""","""T""","""OL""","""O""","""Iowa""",23
2015,1,6,"""NYJ""","""00-0031933""","""WillLe02""","""Leonard Williams""","""DE""","""DL""","""D""","""USC""",21
2015,1,7,"""CHI""","""00-0031545""","""WhitKe00""","""Kevin White""","""WR""","""WR""","""O""","""West Virginia""",23
2015,1,8,"""ATL""","""00-0032240""","""BeasVi00""","""Vic Beasley""","""OLB""","""LB""","""D""","""Clemson""",23
2015,1,9,"""NYG""","""00-0032259""","""FlowEr00""","""Ereck Flowers""","""T""","""OL""","""O""","""Miami (FL)""",21


In [124]:
draft_output = PROCESSED_DIR / "draft_picks_clean.parquet"

draft_picks_clean.write_parquet(draft_output)

print(f"Saved {draft_picks_clean.height:,} draft picks")

Saved 2,821 draft picks


# Clean Depth Chart Data

Depth charts give us another way to look at a player's role on the roster beyond just snap counts.

I want to keep the player, position, and depth chart information so we can eventually use it to measure starters, positional depth, and roster turnover.

In [125]:
depth_charts_clean = (
    depth_charts
    .filter(
        pl.col("season").is_between(2015, 2024) &
        (pl.col("game_type") == "REG")
    )
    .select([
        "season",
        "week",
        pl.col("club_code").alias("team"),
        "gsis_id",
        "full_name",
        "football_name",
        "formation",
        "position",
        "depth_position",
        "depth_team"
    ])
    .sort([
        "season",
        "week",
        "team",
        "depth_position",
        "depth_team"
    ])
)

print(depth_charts_clean.shape)
depth_charts_clean.head(10)

(347064, 10)


season,week,team,gsis_id,full_name,football_name,formation,position,depth_position,depth_team
i32,i32,str,str,str,str,str,str,str,str
2015,1,"""ARI""","""00-0030282""","""Stepfan Taylor""","""Stepfan""","""Offense""","""RB""",""" ""","""3"""
2015,1,"""ARI""","""00-0025682""","""Lyle Sendlein""","""Lyle""","""Offense""","""C""","""C""","""1"""
2015,1,"""ARI""","""00-0026950""","""A.Q. Shipley""","""A.Q.""","""Offense""","""C""","""C""","""2"""
2015,1,"""ARI""","""00-0026190""","""Calais Campbell""","""Calais""","""Defense""","""DE""","""DE""","""1"""
2015,1,"""ARI""","""00-0021998""","""Cory Redding""","""Cory""","""Defense""","""DT""","""DE""","""2"""
2015,1,"""ARI""","""00-0024306""","""Frostee Rucker""","""Frostee""","""Defense""","""DE""","""DT""","""1"""
2015,1,"""ARI""","""00-0030883""","""Josh Mauro""","""Josh""","""Defense""","""DE""","""DT""","""2"""
2015,1,"""ARI""","""00-0031265""","""Ed Stinson""","""Ed""","""Defense""","""DT""","""DT""","""3"""
2015,1,"""ARI""","""00-0030459""","""Tyrann Mathieu""","""Tyrann""","""Defense""","""FS""","""FS""","""1"""


In [126]:
depth_charts_clean = depth_charts_clean.with_columns([
    pl.when(
        pl.col("depth_position").str.strip_chars() == ""
    )
    .then(None)
    .otherwise(pl.col("depth_position").str.strip_chars())
    .alias("depth_position"),

    pl.col("depth_team")
    .cast(pl.Int32)
    .alias("depth_team")
])

In [127]:
depth_charts_output = PROCESSED_DIR / "depth_charts_clean.parquet"

depth_charts_clean.write_parquet(depth_charts_output)

print(f"Saved {depth_charts_clean.height:,} depth chart records")

Saved 347,064 depth chart records


# Clean Contract Data

Contract data gives us another way to measure roster investment, especially at important positions and position groups.

Each row represents a contract rather than a player season, so I am keeping the individual deals instead of reducing players to one record. Current contract status is also kept for reference, but it will not be used to describe past seasons because that could introduce future information.

In [128]:
contract_seasons_clean = (
    contracts
    .select([
        "player",
        "gsis_id",
        "position",
        "season_history"
    ])
    .explode("season_history")
    .unnest("season_history")
    .filter(
        pl.col("year").is_in(
            [str(year) for year in range(2015, 2026)]
        )
    )
    .drop("position")
    .unique()
)

print("Clean contract-season rows:", contract_seasons_clean.height)

contract_seasons_clean.head()

C:\Users\efriedman\AppData\Local\Temp\ipykernel_25768\717881970.py:9: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("season_history")


Clean contract-season rows: 27847


player,gsis_id,year,team,base_salary,prorated_bonus,option_bonus,roster_bonus,guaranteed_salary,cap_number,cap_percent,cash_paid,workout_bonus,per_game_roster_bonus,other_bonus
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Donte Jackson""","""00-0034356""","""2021""","""Panthers""",2.433,0.395005,null,null,0.0,2.828005,0.015,2.433,0.0,0.0,null
"""Bryce Ford-Wheaton""","""00-0038477""","""2023""","""Giants""",0.75,0.006666,null,null,0.216,0.456666,0.002,0.47,null,null,null
"""Al Woods""","""00-0027723""","""2019""","""Seahawks""",1.25,0.4,null,0.3,0.0,2.065441,0.011,2.065441,null,0.3,0.0
"""Eric Ebron""","""00-0031387""","""2017""","""Lions""",1.590337,1.807118,null,0.5,2.090337,3.897455,0.023,2.090337,null,0.0,0.0
"""Don Barclay""","""00-0029187""","""2016""","""Packers""",0.675,0.0,null,null,null,0.7,0.004,0.7,0.025,null,null


In [129]:
CONTRACT_TEAM_MAP = {
    "49ers": "SF",
    "Bears": "CHI",
    "Bengals": "CIN",
    "Bills": "BUF",
    "Broncos": "DEN",
    "Browns": "CLE",
    "Buccaneers": "TB",
    "Cardinals": "ARI",
    "Chargers": "LAC",
    "Chiefs": "KC",
    "Colts": "IND",
    "Commanders": "WAS",
    "Cowboys": "DAL",
    "Dolphins": "MIA",
    "Eagles": "PHI",
    "Falcons": "ATL",
    "Giants": "NYG",
    "Jaguars": "JAX",
    "Jets": "NYJ",
    "Lions": "DET",
    "Packers": "GB",
    "Panthers": "CAR",
    "Patriots": "NE",
    "Raiders": "LV",
    "Rams": "LA",
    "Ravens": "BAL",
    "Redskins": "WAS",
    "Saints": "NO",
    "Seahawks": "SEA",
    "Steelers": "PIT",
    "Texans": "HOU",
    "Titans": "TEN",
    "Vikings": "MIN",
    "Washington": "WAS"
}

In [130]:
contract_seasons_clean = (
    contract_seasons_clean
    .filter(pl.col("team") != "")
    .with_columns([
        pl.col("year").cast(pl.Int32),
        pl.col("team").replace(CONTRACT_TEAM_MAP)
    ])
    .sort([
        "year",
        "team",
        "player"
    ])
)

In [131]:
contract_seasons_output = (
    PROCESSED_DIR / "contract_seasons_clean.parquet"
)

contract_seasons_clean.write_parquet(contract_seasons_output)

print(
    f"Saved {contract_seasons_clean.height:,} "
    "contract-season records"
)

Saved 27,843 contract-season records


# Clean Trade Data

Trade data gives us a way to track player and draft pick movement between teams.

I am keeping trades from 2015 through 2026 so we can measure historical roster movement and also include transactions that have already happened before the 2026 season.

In [132]:
trades_clean = (
    trades
    .filter(pl.col("season").is_between(2015, 2026))
    .select([
        "trade_id",
        "season",
        "trade_date",
        "gave",
        "received",
        "pick_season",
        "pick_round",
        "pick_number",
        "conditional",
        "pfr_id",
        "pfr_name"
    ])
    .sort([
        "trade_date",
        "trade_id"
    ])
)

print(f"Clean trade rows: {trades_clean.height:,}")

trades_clean.head(10)

Clean trade rows: 2,840


trade_id,season,trade_date,gave,received,pick_season,pick_round,pick_number,conditional,pfr_id,pfr_name
i32,i32,date,str,str,i32,i32,i32,i32,str,str
1086,2015,2015-03-10,"""NO""","""SEA""",null,null,null,null,"""GrahJi00""","""Jimmy Graham"""
1086,2015,2015-03-10,"""NO""","""SEA""",2015,4,112,0,"""KouaAr00""","""Arie Koujio"""
1086,2015,2015-03-10,"""SEA""","""NO""",null,null,null,null,"""UngeMa20""","""Max Unger"""
1086,2015,2015-03-10,"""SEA""","""NO""",2015,1,31,0,"""AnthSt00""","""Stephone Anthony"""
1087,2015,2015-03-10,"""PHI""","""LA""",null,null,null,null,"""FoleNi00""","""Nick Foles"""
1087,2015,2015-03-10,"""PHI""","""LA""",2015,4,119,0,"""DonnAn00""","""Andrew Donnal"""
1087,2015,2015-03-10,"""PHI""","""LA""",2016,2,43,0,"""JohnAu01""","""Austin Johnson"""
1087,2015,2015-03-10,"""LA""","""PHI""",null,null,null,null,"""BradSa00""","""Sam Bradford"""
1087,2015,2015-03-10,"""LA""","""PHI""",2015,5,145,0,"""McCaBo01""","""Bobby McCain"""


In [133]:
trades_output = PROCESSED_DIR / "trades_clean.parquet"

trades_clean.write_parquet(trades_output)

print(f"Saved {trades_clean.height:,} trade records")

Saved 2,840 trade records


# Clean Play-by-Play Data

Play-by-play data will be used to build the team efficiency features in the next notebook.

I only need regular season plays from 2015 through 2025, so I am keeping the main game, team, play type, EPA, and success fields that will be useful for feature engineering.

In [134]:
pbp_files = [
    RAW_DIR / f"play_by_play_{season}.parquet"
    for season in SEASONS
]

pbp = pl.concat(
    [
        pl.read_parquet(file)
        for file in pbp_files
    ],
    how="diagonal_relaxed"
)

print(pbp.shape)
print(pbp.columns)

(532376, 372)
['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team', 'season_type', 'week', 'posteam', 'posteam_type', 'defteam', 'side_of_field', 'yardline_100', 'game_date', 'quarter_seconds_remaining', 'half_seconds_remaining', 'game_seconds_remaining', 'game_half', 'quarter_end', 'drive', 'sp', 'qtr', 'down', 'goal_to_go', 'time', 'yrdln', 'ydstogo', 'ydsnet', 'desc', 'play_type', 'yards_gained', 'shotgun', 'no_huddle', 'qb_dropback', 'qb_kneel', 'qb_spike', 'qb_scramble', 'pass_length', 'pass_location', 'air_yards', 'yards_after_catch', 'run_location', 'run_gap', 'field_goal_result', 'kick_distance', 'extra_point_result', 'two_point_conv_result', 'home_timeouts_remaining', 'away_timeouts_remaining', 'timeout', 'timeout_team', 'td_team', 'td_player_name', 'td_player_id', 'posteam_timeouts_remaining', 'defteam_timeouts_remaining', 'total_home_score', 'total_away_score', 'posteam_score', 'defteam_score', 'score_differential', 'posteam_score_post', 'defteam_score_post', 'scor

## Select Play-by-Play Fields

The raw play-by-play files contain hundreds of columns. I am keeping the fields that are most useful for measuring team efficiency, passing and rushing performance, turnovers, situational football, and game context.

In [135]:
play_by_play_clean = (
    pbp
    .filter(
        (pl.col("season").is_between(2015, 2025)) &
        (pl.col("season_type") == "REG")
    )
    .select([
        "game_id",
        "play_id",
        "season",
        "week",
        "home_team",
        "away_team",
        "posteam",
        "defteam",
        "qtr",
        "down",
        "ydstogo",
        "yardline_100",
        "goal_to_go",
        "score_differential",
        "drive",
        "posteam_score",
        "posteam_score_post",
        "play_type",
        "yards_gained",
        "epa",
        "success",
        "wp",
        "wpa",
        "pass_attempt",
        "rush_attempt",
        "qb_dropback",
        "qb_scramble",
        "sack",
        "qb_hit",
        "complete_pass",
        "interception",
        "fumble",
        "fumble_lost",
        "touchdown",
        "pass_touchdown",
        "rush_touchdown",
        "air_yards",
        "yards_after_catch",
        "cpoe",
        "xpass",
        "pass_oe",
        "first_down",
        "third_down_converted",
        "third_down_failed",
        "fourth_down_converted",
        "fourth_down_failed",
        "shotgun",
        "fixed_drive",
        "fixed_drive_result",
        "drive_ended_with_score",
        "no_huddle"
    ])

    .sort([
        "season",
        "week",
        "game_id",
        "play_id"
    ])
)

print(play_by_play_clean.shape)
play_by_play_clean.head(10)

(508914, 51)


game_id,play_id,season,week,home_team,away_team,posteam,defteam,qtr,down,ydstogo,yardline_100,goal_to_go,score_differential,drive,posteam_score,posteam_score_post,play_type,yards_gained,epa,success,wp,wpa,pass_attempt,rush_attempt,qb_dropback,qb_scramble,sack,qb_hit,complete_pass,interception,fumble,fumble_lost,touchdown,pass_touchdown,rush_touchdown,air_yards,yards_after_catch,cpoe,xpass,pass_oe,first_down,third_down_converted,third_down_failed,fourth_down_converted,fourth_down_failed,shotgun,fixed_drive,fixed_drive_result,drive_ended_with_score,no_huddle
str,f64,i32,i32,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64
"""2015_01_BAL_DEN""",1.0,2015,1,"""DEN""","""BAL""",null,null,1.0,null,0.0,null,0.0,null,null,null,null,null,null,-0.0,0.0,0.422024,-0.0,null,null,null,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,1.0,"""Punt""",null,0.0
"""2015_01_BAL_DEN""",36.0,2015,1,"""DEN""","""BAL""","""BAL""","""DEN""",1.0,null,0.0,35.0,0.0,0.0,1.0,0.0,0.0,"""kickoff""",0.0,-0.0,0.0,0.422024,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,0.0,0.0,0.0,0.0,0.0,0.0,1.0,"""Punt""",0.0,0.0
"""2015_01_BAL_DEN""",51.0,2015,1,"""DEN""","""BAL""","""BAL""","""DEN""",1.0,1.0,10.0,80.0,0.0,0.0,1.0,0.0,0.0,"""pass""",3.0,-0.337139,0.0,0.422024,-0.001425,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,23.418927,0.456481,54.351911,0.0,0.0,0.0,0.0,0.0,0.0,1.0,"""Punt""",0.0,0.0
"""2015_01_BAL_DEN""",75.0,2015,1,"""DEN""","""BAL""","""BAL""","""DEN""",1.0,2.0,7.0,77.0,0.0,0.0,1.0,0.0,0.0,"""run""",2.0,-0.262481,0.0,0.420599,-0.017304,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,0.545905,-54.590458,0.0,0.0,0.0,0.0,0.0,0.0,1.0,"""Punt""",0.0,0.0
"""2015_01_BAL_DEN""",96.0,2015,1,"""DEN""","""BAL""","""BAL""","""DEN""",1.0,3.0,5.0,75.0,0.0,0.0,1.0,0.0,0.0,"""pass""",10.0,1.661242,1.0,0.403295,0.045358,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,6.0,48.982388,0.968533,3.146732,1.0,1.0,0.0,0.0,0.0,1.0,1.0,"""Punt""",0.0,0.0
"""2015_01_BAL_DEN""",120.0,2015,1,"""DEN""","""BAL""","""BAL""","""DEN""",1.0,1.0,10.0,65.0,0.0,0.0,1.0,0.0,0.0,"""run""",0.0,-0.518931,0.0,0.448653,-0.018066,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,0.448571,-44.857121,0.0,0.0,0.0,0.0,0.0,0.0,1.0,"""Punt""",0.0,0.0
"""2015_01_BAL_DEN""",141.0,2015,1,"""DEN""","""BAL""","""BAL""","""DEN""",1.0,2.0,10.0,65.0,0.0,0.0,1.0,0.0,0.0,"""pass""",3.0,-0.449598,0.0,0.430587,-0.021081,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,22.074765,0.489852,51.014823,0.0,0.0,0.0,0.0,0.0,0.0,1.0,"""Punt""",0.0,0.0
"""2015_01_BAL_DEN""",165.0,2015,1,"""DEN""","""BAL""","""BAL""","""DEN""",1.0,3.0,7.0,62.0,0.0,0.0,1.0,0.0,0.0,"""pass""",0.0,-1.469601,0.0,0.409506,-0.027282,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,null,-67.080569,0.979308,2.069181,0.0,0.0,1.0,0.0,0.0,1.0,1.0,"""Punt""",0.0,0.0
"""2015_01_BAL_DEN""",187.0,2015,1,"""DEN""","""BAL""","""BAL""","""DEN""",1.0,4.0,7.0,62.0,0.0,0.0,1.0,0.0,0.0,"""punt""",0.0,0.620374,1.0,0.382224,0.014567,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,0.0,0.0,0.0,0.0,0.0,0.0,1.0,"""Punt""",0.0,0.0


In [136]:
play_by_play_output = (
    PROCESSED_DIR / "play_by_play_clean.parquet"
)

play_by_play_clean.write_parquet(play_by_play_output)

print(
    f"Saved {play_by_play_clean.height:,} "
    "play-by-play records"
)

Saved 508,914 play-by-play records


# Final Check

Before moving into feature engineering, I want to make sure the cleaned datasets were created successfully and still have the basic structure I expect.

In [137]:
cleaned_data = {
    "Schedules": schedule_clean,
    "Season Rosters": rosters_clean,
    "Player Stats": player_stats_clean,
    "Snap Counts": snap_counts_clean,
    "Participation": participation_clean,
    "Injuries": injuries_clean,
    "Weekly Rosters": weekly_rosters_clean,
    "Draft Picks": draft_picks_clean,
    "Depth Charts": depth_charts_clean,
    "Contracts": contract_seasons_clean,
    "Play-by-Play": play_by_play_clean,
    "Trades": trades_clean
}

for name, df in cleaned_data.items():
    print(f"{name}: {df.height:,} rows")

Schedules: 2,895 rows
Season Rosters: 33,195 rows
Player Stats: 21,366 rows
Snap Counts: 264,774 rows
Participation: 429,967 rows
Injuries: 58,447 rows
Weekly Rosters: 475,749 rows
Draft Picks: 2,821 rows
Depth Charts: 347,064 rows
Contracts: 27,843 rows
Play-by-Play: 508,914 rows
Trades: 2,840 rows


In [138]:
VALID_TEAMS = {
    "ARI", "ATL", "BAL", "BUF", "CAR", "CHI", "CIN", "CLE",
    "DAL", "DEN", "DET", "GB", "HOU", "IND", "JAX", "KC",
    "LA", "LAC", "LV", "MIA", "MIN", "NE", "NO", "NYG",
    "NYJ", "PHI", "PIT", "SEA", "SF", "TB", "TEN", "WAS"
}

assert set(schedule_clean["home_team"].unique()).issubset(VALID_TEAMS)
assert set(schedule_clean["away_team"].unique()).issubset(VALID_TEAMS)
assert set(rosters_clean["team"].unique()).issubset(VALID_TEAMS)
assert set(player_stats_clean["recent_team"].unique()).issubset(VALID_TEAMS)
assert set(snap_counts_clean["team"].unique()).issubset(VALID_TEAMS)

print("Final team check passed.")

Final team check passed.
